# Stage1 cap-release action-value source preflight

Frozen source/support audit only. No future returns, labels, fits or accounts.


In [1]:
import hashlib, json, os
from pathlib import Path
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R

root=Path.cwd() if (Path.cwd()/'research/stage1_cap_release_value_preflight_20261023/plan.json').exists() else Path.cwd().parents[1]; card=root/'research/stage1_cap_release_value_preflight_20261023'
plan=json.loads((card/'plan.json').read_text())
repair=json.loads((card/'formula_repair_plan.json').read_text())
assert repair['frozen'] and repair['original_plan_sha256']==hashlib.sha256((card/'plan.json').read_bytes()).hexdigest()
assert repair['invalid_2x_result_sha256']==hashlib.sha256((card/'invalid_2x_result.json').read_bytes()).hexdigest()
assert plan['frozen'] and all(v==0 for k,v in plan['budget'].items() if k!='source_recorder_reads')
sha=lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
s=plan['source']
for key,hashkey in [('training_card','training_summary_sha256'),('held_risk_gap_summary','held_risk_gap_summary_sha256'),('held_loss_chain_summary','held_loss_chain_summary_sha256'),('closed_h5_dual_receiver_summary','closed_h5_dual_receiver_summary_sha256')]:
    path=root/s[key]/'summary.json' if key=='training_card' else root/s[key]
    assert sha(path)==s[hashkey], (key,sha(path))
training=json.loads((root/s['training_card']/'summary.json').read_text())
assert training['recorder_id']==s['training_recorder_id'] and training['experiment_id']==s['training_experiment_id']
assert training['nas_readback']['verified'] and training['nas_readback']['sha256']==training['nas_evidence']['sha256']
provider=(root/s['provider_uri']).resolve(); tracking=(root/s['training_card']/'qlib_recorder').resolve()
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
qlib.init(provider_uri=str(provider),region='us',exp_manager={'class':'MLflowExpManager','module_path':'qlib.workflow.expm','kwargs':{'uri':str(tracking),'default_exp_name':Path(s['training_card']).name}})
rec=R.get_recorder(experiment_name=Path(s['training_card']).name,recorder_id=s['training_recorder_id'])
frame=rec.load_object('inputs/source_frame')
path=rec.load_object('paths/reference_0')['replay']
assert len(frame)==len(path)==1061
f=frame[['trade_date','signal_date','decision_at','feature_available_at','soxl_weight','soxx_weight','cash_weight']].copy()
p=path[['trade_date','current_beta_cap','risk_state','execute_open_target']].copy()
assert f.trade_date.astype(str).tolist()==p.trade_date.astype(str).tolist()
cal=pd.to_datetime(D.calendar(start_time='2018-10-15',end_time='2022-12-30',freq='day')).date
assert len(cal)==1061 and list(cal)==list(pd.to_datetime(f.trade_date).dt.date)
for c in ['soxl_weight','soxx_weight','cash_weight','current_beta_cap']:
    values=f[c] if c in f else p[c]
    assert np.isfinite(pd.to_numeric(values,errors='coerce')).all(),c
assert (pd.to_datetime(f.feature_available_at)<=pd.to_datetime(f.decision_at)).all()
raw=3*f.soxl_weight.to_numpy(float)+f.soxx_weight.to_numpy(float)
cap=p.current_beta_cap.to_numpy(float); cash=f.cash_weight.to_numpy(float)
gap=raw-cap
binding=gap>1e-9
structural=binding & (gap>=0.10) & (cash>=0.10)
years=pd.to_datetime(f.trade_date).dt.year
annual=[{'year':int(y),'rows':int((years==y).sum()),'cap_binding_rows':int((binding & (years==y)).sum()),'structural_010_rows':int((structural & (years==y)).sum())} for y in sorted(years.unique())]
result={'decision':'source_support_only_no_label_fit_or_account','corrected_formula':'3*SOXL_weight+SOXX_weight','invalid_2x_result':'invalid_2x_result.json','source_recorder_id':rec.id,'provider_uri':str(provider),'sessions':len(f),'qlib_calendar_same_order':True,'source_feature_clock_precedes_decision':True,'finite_support_inputs':True,'cap_binding_rows':int(binding.sum()),'structural_010_rows':int(structural.sum()),'annual':annual,'new_market_requests':0,'new_model_fits':0,'new_native_accounts':0,'new_labels':0,'new_bootstraps':0,'interpretation':'Structural source support only. The saved OOF training reference is 2018-2022 and not the frozen 2023+ f696+b472 held own-state account. Raw source weights and cap do not prove a funded 0.10-beta SOXX order can execute or improve full-account wealth. A physical fee-net same-state label and receiver require a separate frozen plan.'}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'formula_repair_plan_sha256':sha(card/'formula_repair_plan.json'),'invalid_2x_result_sha256':sha(card/'invalid_2x_result.json'),'result':'result.json','source_recorder_id':rec.id,'source_experiment_id':training['experiment_id'],'source_NAS_sha256':training['nas_evidence']['sha256'],'source_NAS_readback_verified':True,'budget_used':{k:0 for k in plan['budget'] if k!='source_recorder_reads'},'source_recorder_reads':2,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':[]}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(json.dumps(result,ensure_ascii=False))

[27364:MainThread](2026-10-23 08:08:09,702) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[27364:MainThread](2026-10-23 08:08:09,703) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[27364:MainThread](2026-10-23 08:08:09,704) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


{"decision": "source_support_only_no_label_fit_or_account", "corrected_formula": "3*SOXL_weight+SOXX_weight", "invalid_2x_result": "invalid_2x_result.json", "source_recorder_id": "3663a1e073f042e383dbcf54969a9bc2", "provider_uri": "/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013", "sessions": 1061, "qlib_calendar_same_order": true, "source_feature_clock_precedes_decision": true, "finite_support_inputs": true, "cap_binding_rows": 541, "structural_010_rows": 541, "annual": [{"year": 2018, "rows": 53, "cap_binding_rows": 18, "structural_010_rows": 18}, {"year": 2019, "rows": 252, "cap_binding_rows": 39, "structural_010_rows": 39}, {"year": 2020, "rows": 253, "cap_binding_rows": 182, "structural_010_rows": 182}, {"year": 2021, "rows": 252, "cap_binding_rows": 56, "structural_010_rows": 56}, {"year": 2022, "rows": 251, "cap_binding_rows": 246, "structural_010_rows": 246}], "new_market_requests": 0, "new_model_fits": 0, "new_native_accounts": 0, "new_labels": 0, "

# Stage1 风险上限释放价值：只读来源预检

目标是判断旧训练期是否有足够的决策前风险上限约束与现金，供后续单独设计“放松上限并增持 SOXX”的费用后局部价值标签。来源是已保存的 2018–2022 Stage1 原架构 OOF 0bp Tiered 训练参考 Recorder；它不是冻结的 2023+ f696+b472 自有资金账户。

首次预检误把 SOXL 权重按 2 倍折算为 beta，得出的 519/517 日**无效**。原始冻结计划、错误结果及其哈希保留在本卡。机械修正计划预先固定正确公式 `3 × SOXL 权重 + SOXX 权重`，只重算同一来源支持度。Qlib 日历与来源逐日同序共 **1061** 日，特征可用时间均不晚于决策时间，输入全为有限值。原始 beta 超过自身风险上限 **541** 日；这些日子同时满足预先固定的至少 0.10 beta 缺口和 0.10 来源现金权重。年分布为 2018:18、2019:39、2020:182、2021:56、2022:246。2019/2021 较薄，合计数量不能当作跨年预测稳定性的证据。

此处没有读取未来收益、形成局部标签、拟合模型、运行候选账户或请求新行情。来源现金权重与上限只说明结构支持，不能证明原始整股订单可执行，也不能证明费用后收益。若继续，须另行冻结同状态、相同下一目标复位、原始整股与 IBKR Tiered 费用的局部标签和按年信息门，再运行完整 Stage1 衍生 0bp 自有资金账户。已关闭的五日现金与相对收益信息版本不会因本预检重开。SOXX 目标尚未实现。
